## Generator simple class
Still using code from Master thesis for data processing as well as processed dataset which was already processed using the extract_adult dataset

In [1]:
import os
import sys
from pathlib import Path
import random

sys.path.append('..')
sys.path.append('../libs/MIA-synthetic-main')
# os.environ['OMP_PATH'] = '/opt/homebrew/Cellar/libomp/19.1.3/include'
os.environ["OMP_PATH"] = r"C:\Path\To\OpenMP\include"

# third-party
import pandas as pd
import numpy as np

from tapas.generators.generator import ReprosynGenerator
from reprosyn.methods import DS_PRIVBAYES

from tools.tapas.tapas_data_processors import AdultDataProcessor
from tapas.datasets import TabularDataset

In [2]:
os.getcwd()

'c:\\Users\\Utente\\OneDrive\\Documenti\\GitHub\\Benchmarking-of-Tabular-Synthetic-Data-Generation\\privacy\\notebooks'

In [3]:
os.chdir('../')

In [4]:
DATASET_NAME = 'adult'
RANDOM_STATE=42
gen = ReprosynGenerator(DS_PRIVBAYES, label="PrivBayes", seed=RANDOM_STATE, epsilon=1.0)

# reproducibility
np.random.seed(RANDOM_STATE)
pd.np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

# Load the dataset
DATA_FOLDER = Path('./data')
DATA_PATH = Path(f'./data/{DATASET_NAME}')
full_ds = TabularDataset.read(DATA_PATH, label=DATASET_NAME)
full_ds = AdultDataProcessor.process_tapas_tabulardataset(full_ds)
categorical_features = full_ds.description.one_hot_cols
numerical_features = [col for col in full_ds.description.columns if col not in categorical_features]

# pre-define test samples for ML utility
N_SYNTH_SAMPLES = 1000
N_TEST_SAMPLES = 200

np_rng = np.random.default_rng(RANDOM_STATE)
record_ids = np_rng.integers(0, len(full_ds.data), N_SYNTH_SAMPLES+N_TEST_SAMPLES)
train_records_ids = record_ids[:N_SYNTH_SAMPLES]
test_records_ids = record_ids[N_SYNTH_SAMPLES:]
test_ds = full_ds.get_records(test_records_ids)
train_ds = full_ds.get_records(train_records_ids)

C:\Users\Utente\AppData\Local\Temp\ipykernel_7896\513580146.py:7: FutureWarning: The pandas.np module is deprecated and will be removed from pandas in a future version. Import numpy directly instead.
  pd.np.random.seed(RANDOM_STATE)


In [5]:
gen.fit(train_ds)
synth_data = gen.generate(N_SYNTH_SAMPLES)

In [10]:
synth_data

In [9]:
synth_data.data

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,33,Private,6.189290e+05,HS-grad,1,Widowed,Craft-repair,Unmarried,Black,Male,38640.120800,922.390147,61.071189,Honduras,<=50K
1,72,Self-emp-inc,1.148596e+06,HS-grad,9,Separated,Tech-support,Unmarried,Other,Male,36509.330534,2823.664151,26.113817,Vietnam,>50K
2,56,Private,8.323586e+05,10th,4,Married-civ-spouse,Sales,Wife,Amer-Indian-Eskimo,Female,16871.435326,2234.771243,25.693117,Laos,<=50K
3,47,Self-emp-not-inc,1.579103e+05,Preschool,9,Married-civ-spouse,Farming-fishing,Not-in-family,White,Male,1626.548852,2005.273558,65.992205,United-States,<=50K
4,25,Local-gov,8.049188e+05,Prof-school,10,Never-married,Sales,Husband,White,Male,737.086525,796.068487,25.426544,Vietnam,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,72,Private,5.036635e+05,5th-6th,10,Divorced,Transport-moving,Unmarried,Other,Male,34758.344268,4185.612711,72.732875,Vietnam,>50K
996,39,Self-emp-not-inc,6.925621e+05,Bachelors,10,Divorced,Farming-fishing,Own-child,White,Male,96146.520220,2751.462416,68.407509,United-States,<=50K
997,49,Private,7.035820e+05,5th-6th,14,Divorced,Craft-repair,Unmarried,Asian-Pac-Islander,Male,70389.661733,228.860418,41.527874,Dominican-Republic,>50K
998,9,Local-gov,7.993053e+05,Some-college,8,Married-civ-spouse,Armed-Forces,Own-child,Black,Female,97837.703962,433.432601,32.000427,Portugal,<=50K
